In [0]:
%pip install -q databricks-langchain "langgraph>=0.3,<0.4" databricks-agents "psycopg[binary]>=3.1.0" pydantic databricks-ai-search

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Fraud Triage Copilot Agent
**Step 5:** Fraud Triage Copilot (Layer 4) + Lakebase FGAC

- FM endpoint: `databricks-claude-sonnet-5-5`
- Tools (since 2026-10-21 FGAC hardening): all agent-side and Postgres-governed via `LakebaseCases(pg_role='fraud_triage_agent')`: `get_transaction_risk` (calibrated confidence), `explain_top_signals`, `get_customer_history`, `create_fraud_case`. The UC function versions are deprecated and not registered.
- Registered: `workshop_guy_catalog.fe_bar_a0.fraud_triage_agent` v1 @champion
- Experiment: `/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent`

In [0]:
import sys, json, time
sys.path.insert(0, "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")

import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent")

from fraud_triage_agent import FraudTriageAgent
from mlflow.types.responses import ResponsesAgentRequest

agent = FraudTriageAgent()
print("Agent ready.")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026/11/01 19:07:41 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or 

Agent ready.


In [0]:
# E.1: Why was TXN-0234209 flagged?
request = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-0234209 flagged and what should we do?"}]
)
t0 = time.time()
response = agent.predict(request)
print(f"Latency: {time.time()-t0:.1f}s")
print(response.output[0].content[0]["text"])

# FGAC hardening Item 5 VERIFY: trace span table for this run.
trace_id = mlflow.get_last_active_trace_id()
trace = mlflow.get_trace(trace_id)
print(f"\n=== Trace span table  trace_id={trace_id} ===")
rows = []
for s in trace.data.spans:
    if not str(s.span_type).upper().endswith("TOOL"):
        continue
    out = s.outputs
    if hasattr(out, "content"): out = out.content
    if isinstance(out, dict) and "content" in out: out = out["content"]
    try:
        payload = json.loads(out) if isinstance(out, str) else {}
    except Exception:
        payload = {}
    rows.append((s.name, payload.get("_implementation", "(no provenance)"), ", ".join(payload.get("_objects_read", [])), (s.end_time_ns - s.start_time_ns) / 1e6))
print(f"{'tool span':<24} {'implementation':<42} {'ms':>8}  objects read")
print("-" * 150)
for name, impl, objs, ms in rows:
    print(f"{name:<24} {impl:<42} {ms:>8.1f}  {objs}")
assert rows, "no tool spans found"
assert all(r[1] == "agent_python:postgres:fraud_triage_agent" for r in rows), "a tool ran outside Postgres governance"
print(f"\n{len(rows)} tool spans, all Postgres-governed (fraud_triage_agent); no UC function tool was called.")

2026/11/01 19:07:45 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's... new one if you want.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Latency: 16.0s
## TXN-0234209: $323.96, CUST-08669

**Why it was flagged**
The risk score is **0.984**. Three signals drive it:
- **New device (SHAP +2.57):** This is the strongest signal. The customer normally uses about 5 devices a month, so a new device is less unusual for them than for most customers.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions score higher for risk.

**Recommended action:** **Block.** This is the model's recommendation.

**Confidence:** **Medium (observed precision 0.50 in block_band, n=288).** The score is very high, but historically only about half of the transactions in this band were confirmed fraud. Treat the score as a strong flag, not a near-certain one.

**Routed queue:** `auto_block`

**Customer context**
- The customer has a low risk segment, 40 months of tenure, and a US-East home region.
- In the last 30 days they made 12 transactions avera

Trace(trace_id='tr-372bc4fc3a27c07127bdc96756ac3920')

In [0]:
# E.2: Open a case
request_act = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
)
t0 = time.time()
response_act = agent.predict(request_act)
print(f"Latency: {time.time()-t0:.1f}s")
print(response_act.output[0].content[0]["text"])
# Case 2319d4b7-bea3-412b-ba6b-8325e1a7a71d (opened in Step 5) is still open, so the
# idempotent create_fraud_case should return that same case_id.
from lakebase_cases import LakebaseCases
_lc = LakebaseCases()
for c in _lc.get_cases_for_txn("TXN-0234209"):
    print(f"case {c['case_id']} status={c['status']} queue={c['assigned_queue']} created_by={c['created_by']} db_inserted_by={c['db_inserted_by']}")
_lc.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...Wd1TRsHTFF3iV9J9xgB'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...xisting one as it is?"}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Latency: 10.6s
I haven't created a new case, because TXN-0234209 already has one open.

**Existing case:** `2319d4b7-bea3-412b-ba6b-8325e1a7a71d`, status **open**, queue **auto_block**. A second case would duplicate it.

## TXN-0234209: $323.96, CUST-08669

**Why flagged** (risk score 0.984)
- **New device (SHAP +2.57):** the customer normally uses 5 devices in 30 days.
- **450.6 km from home (SHAP +1.87):** the home region is US-East.
- **Online transaction (SHAP +1.20):** this channel carries higher risk.

**Recommended action:** block.

**Confidence:** medium (observed precision 0.50 in block_band, n=288). Only about half of transactions in this band turn out to be fraud, so a human should review the block.

**Routed queue:** auto_block.

**Customer context**
- Low-risk segment, 40 months of tenure, home region US-East.
- Prior history: 3 cases, all 3 confirmed fraud.
- In the last 30 days: 12 transactions averaging about $100, 5 devices, 4 countries and 5 cross-border transactions.

Trace(trace_id='tr-2e573295e00f2d579ae2cc85618c2abe')

In [0]:
# E.3: Ask again - should return same case_id
response_idem = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
))
print(response_idem.output[0].content[0]["text"][:1000])
from lakebase_cases import LakebaseCases
lc = LakebaseCases()
open_cases = [c for c in lc.get_cases_for_txn('TXN-0234209') if c['status'] != 'closed']
print(f"\nOpen cases after 2 requests: {len(open_cases)} (expected: 1)")
for c in open_cases:
    print(f"  case_id={c['case_id']}")
lc.close()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...9uHfCJ1mdQn/UsyGAE='}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...del's action (block)."}], input_type=list])
  return self.__pydantic_serializer__.to_python(


I haven't opened a new case for TXN-0234209. An open case already exists for it, and I didn't want to create a duplicate.

**Existing case:** `2319d4b7-bea3-412b-ba6b-8325e1a7a71d`. Its status is open and it's in the `auto_block` queue.

## TXN-0234209: $323.96, CUST-08669
**Why flagged**
- **New device (SHAP +2.57):** The customer normally uses 5 devices in 30 days, and this one is new.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the home region, US-East.
- **Online transaction (SHAP +1.20):** Online transactions score as higher risk.

**Recommended action:** Block. The risk score is 0.984.

**Confidence:** Medium (observed precision 0.50 in block_band, n=288). Only about half of past transactions in this band were confirmed fraud, so the high score alone overstates how sure the model is.

**Routed queue:** `auto_block`

**Customer context**
- The customer is low-risk, with 40 months of tenure and a home region of US-East.
- They have 3 prior cases, and al

Trace(trace_id='tr-85307d847f0205ef7832b13288f36c02')

## E.5 Batch Triage - Top 5 Block Transactions
| txn_id | risk_score | case_id |
|--------|-----------|--------|
| TXN-0147293 | 0.993762 | 82b12d25-bf88-47cb-b0a4-f5a5b62f349d |
| TXN-0288343 | 0.992881 | 863334d8-909e-4348-8be4-772d0c7f6e72 |
| TXN-0221786 | 0.992875 | c37587c0-5091-4fd7-a656-c9df9417d0e3 |
| TXN-0417769 | 0.992698 | 98edced4-9dc7-47a3-bbf3-1694bb7b1b43 |
| TXN-0432087 | 0.992657 | 98adcae4-cec6-46a1-a2c5-1ecd84c299ad |

In [0]:
# E.6: Guardrails
# Unknown txn
resp_unknown = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-9999999 flagged?"}]
))
print("--- Unknown txn ---")
print(resp_unknown.output[0].content[0]["text"][:500])

# Approve band should not be blocked
resp_approve = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Block TXN-0278800 immediately."}]
))
print("\n--- Approve-band ---")
print(resp_approve.output[0].content[0]["text"][:500])

--- Unknown txn ---
I couldn't find **TXN-9999999**. Both the risk lookup and the signal-explanation lookup came back empty. That means I have no score, recommended action, confidence level, routed queue, or SHAP signals for it. I can't say why it was flagged, and I won't guess.

Here is what you could check:
- **Typo or format:** Confirm the ID is exactly right, including the prefix and digit count.
- **Scoring status:** The transaction may not have been scored yet, or it may be outside the current risk tables.
- 


/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...is transaction first."}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...Which would you like?"}], input_type=list])
  return self.__pydantic_serializer__.to_python(



--- Approve-band ---
I haven't blocked TXN-0278800. The model scored it in the approve band, and I shouldn't push an approve-band transaction into block. Tell me how you want to proceed and I'll open the case.

**Why it was flagged**
- **New device (SHAP +1.76):** This is the strongest signal. The customer normally uses about 5 devices.
- **Online transaction (SHAP +0.99):** Online transactions carry higher risk.
- **Distance from home (SHAP +0.62):** The transaction was 575.6 km from the customer's EU home region.



[Trace(trace_id='tr-a88587ca4a26ed9111b7f2923c1966a0'), Trace(trace_id='tr-c03d76d0b0368e0599a8f3bbc07fdf8b')]

In [0]:
# E.7 (Step 7 Part E VERIFY): explain_top_signals previously raised
# "Unknown format code 'f' for object of type 'str'" for TXN-0112255 (a SHAP signal value was a string).
# Read-only: calls the tool directly against fraud_gov.* as fraud_triage_agent. Writes nothing.
from fraud_triage_agent import explain_top_signals
for txn in ["TXN-0112255", "TXN-0234209"]:
    out = json.loads(explain_top_signals.invoke({"txn_id": txn}))
    rows = out.get("rows") or out.get("content") or out
    print(f"=== {txn}: OK, implementation={out.get('_implementation')} ===")
    for r in (rows if isinstance(rows, list) else [rows]):
        print(" ", r)
print("\nexplain_top_signals returned for both transactions with no format error.")

=== TXN-0112255: OK, implementation=agent_python:postgres:fraud_triage_agent ===
  {'txn_id': 'TXN-0112255', 'signal_name': 'is_new_device', 'shap_value': 1.9395, 'feature_value': '1', 'customer_normal': 'Customer uses 5 devices in 30d', 'explanation': 'New device (SHAP +1.94). Normally uses 5 devices.'}
  {'txn_id': 'TXN-0112255', 'signal_name': 'geo_distance_from_home', 'shap_value': 1.8503, 'feature_value': '332.9 km', 'customer_normal': 'Home region: US-East', 'explanation': '332.9 km from home (US-East), SHAP +1.85.'}
  {'txn_id': 'TXN-0112255', 'signal_name': 'risk_segment', 'shap_value': 0.8605, 'feature_value': 'high', 'customer_normal': 'See customer profile', 'explanation': 'risk_segment = high (SHAP +0.86)'}
=== TXN-0234209: OK, implementation=agent_python:postgres:fraud_triage_agent ===
  {'txn_id': 'TXN-0234209', 'signal_name': 'is_new_device', 'shap_value': 2.5724, 'feature_value': '1', 'customer_normal': 'Customer uses 5 devices in 30d', 'explanation': 'New device (SHAP 

[Trace(trace_id='tr-e4020b760fc258d0b4ca95939c1f447b'), Trace(trace_id='tr-535fd86d5a7be72617009dbf84936fe6')]

In [0]:
# D.3: Log and register to UC
# GUARD (2026-10-21 hardening): v1 @champion was registered in Step 5 with the now-deprecated UC function
# tools. Re-registering creates v2 and moves @champion, so it runs only when explicitly enabled.
# Safety flag: False = no new agent version is logged or registered, @champion is not moved.
# 2026-11-01 (Step 7 follow-up): run once with REGISTER_AGENT = True to register v3 from the current
# fraud_triage_agent.py (Postgres-governed tools + explain_top_signals fix). v2 is kept. Not deployed.
REGISTER_AGENT = False  # was True for the 2026-11-01 19:34 UTC run shown below (registered v3); reset without re-running
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent")
FULL_NAME = "workshop_guy_catalog.fe_bar_a0.fraud_triage_agent"
REPO = "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo"
mc = MlflowClient(registry_uri="databricks-uc")

if not REGISTER_AGENT:
    v = mc.get_model_version_by_alias(FULL_NAME, "champion")
    print(f"Registration skipped (REGISTER_AGENT=False). Current @champion: v{v.version}, run {v.run_id}.")
else:
    import subprocess, sys
    from mlflow.models.resources import DatabricksServingEndpoint
    before = mc.get_model_version_by_alias(FULL_NAME, "champion")
    print(f"BEFORE: @champion -> v{before.version} (run {before.run_id})")
    # Declared resources = what the agent uses now. The read/write tools run in agent Python against
    # Lakebase Postgres (project fe-bar-fraud-analytics, branch production, endpoint primary, db fe_bar_db).
    # No deprecated UC functions. Lakebase is NOT declared: mlflow.models.resources.DatabricksLakebase only
    # takes a provisioned database_instance_name, and this Lakebase Autoscaling project has no database instance
    # (w.database.list_database_instances() is empty). Recorded as a known limitation; agent is not deployed.
    resources = [DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-5-5")]
    input_example = {"input": [{"role": "user", "content": "Why was TXN-0234209 flagged?"}]}
    with mlflow.start_run(run_name="fraud_triage_agent_v3_2026-11-01") as run:
        mlflow.set_tags({"lakebase_endpoint": "projects/fe-bar-fraud-analytics/branches/production/endpoints/primary",
                         "lakebase_database": "fe_bar_db", "pg_role": "fraud_triage_agent",
                         "lakebase_resource_declared": "false (DatabricksLakebase needs a database_instance_name)"})
        info = mlflow.pyfunc.log_model(
            name="agent",
            python_model=f"{REPO}/fraud_triage_agent.py",
            code_paths=[f"{REPO}/lakebase_cases.py"],  # imported by the agent's tools
            resources=resources,
            input_example=input_example,
            pip_requirements=["mlflow>=3.1", "databricks-sdk", "databricks-langchain", "langgraph>=0.3,<0.4",
                              "databricks-agents", "pydantic>=2", "psycopg[binary]>=3.1.0", "databricks-ai-search"],
        )
    print(f"logged: run_id={run.info.run_id} model_uri={info.model_uri}")
    print("declared resources:", [f"{type(r).__name__}({r.name})" for r in resources])

    # Gate before registration: signature + input example, then an isolated uv env installs
    # pip_requirements from scratch, loads the model and predicts on the stored input example.
    loaded = mlflow.pyfunc.load_model(info.model_uri)
    assert loaded.metadata.signature is not None and loaded.input_example is not None
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    mlflow.models.predict(model_uri=info.model_uri, input_data=loaded.input_example, env_manager="uv")
    print("isolated uv validation: PASSED")

    reg = mlflow.register_model(model_uri=info.model_uri, name=FULL_NAME, await_registration_for=300)
    mc.set_registered_model_alias(FULL_NAME, "champion", reg.version)
    after = mc.get_model_version_by_alias(FULL_NAME, "champion")
    kept = mc.get_model_version(FULL_NAME, before.version)
    print(f"Registered: {FULL_NAME} v{reg.version} (run {run.info.run_id})")
    print(f"AFTER: @champion moved v{before.version} -> v{after.version}; v{kept.version} kept (status {kept.status}, aliases {kept.aliases})")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


BEFORE: @champion -> v2 (run 97bc6c9b4e654abab627cd25ee93fa53)


2026/11/01 19:34:25 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
2026/11/01 19:34:26 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://fevm-workshop-guy.cloud.databricks.com/ml/experiments/1507503687185844/models/m-11f97010a94b494aa60c3185dcd340f1?o=7474659392412718
2026/11/01 19:34:26 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.
2026/11/01 19:34:26 INFO mlflow.pyfunc: Predicting on input example to validate output
/local_disk0

logged: run_id=d25dfe72e129478a8e17c0d6dbc7d433 model_uri=models:/m-11f97010a94b494aa60c3185dcd340f1
declared resources: ['DatabricksServingEndpoint(databricks-claude-sonnet-5-5)']


2026/11/01 19:35:02 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.


2026/11/01 19:35:08 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


... (2 outputs left out)

Registered model 'workshop_guy_catalog.fe_bar_a0.fraud_triage_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workshop_guy_catalog.fe_bar_a0.fraud_triage_agent': https://fevm-workshop-guy.cloud.databricks.com/explore/data/models/workshop_guy_catalog/fe_bar_a0/fraud_triage_agent/version/3?o=7474659392412718


Registered: workshop_guy_catalog.fe_bar_a0.fraud_triage_agent v3 (run d25dfe72e129478a8e17c0d6dbc7d433)
AFTER: @champion moved v2 -> v3; v2 kept (status READY, aliases [])


In [0]:
# D.4 (Step 7 follow-up VERIFY): load the registered @champion and explain two transactions.
# Explain only: the prompt asks not to open cases; case counts are checked before/after.
import json, time, mlflow, datetime
from mlflow.tracking import MlflowClient
from lakebase_cases import LakebaseCases
mlflow.set_registry_uri("databricks-uc")
FULL_NAME = "workshop_guy_catalog.fe_bar_a0.fraud_triage_agent"
champ = MlflowClient(registry_uri="databricks-uc").get_model_version_by_alias(FULL_NAME, "champion")
print(f"run_at_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}")
print(f"@champion -> v{champ.version} (run {champ.run_id})")
model = mlflow.pyfunc.load_model(f"models:/{FULL_NAME}@champion")

TXNS = ["TXN-0234209", "TXN-0112255"]
def case_ids():
    lc = LakebaseCases(); out = {t: sorted(c["case_id"] for c in lc.get_cases_for_txn(t)) for t in TXNS}; lc.close(); return out
cases_before = case_ids()

results = []
for txn in TXNS:
    t0 = time.time()
    resp = model.predict({"input": [{"role": "user", "content": f"Explain why {txn} was flagged. Explain only: do not open, update or close any case."}]})
    text = resp["output"][0]["content"][0]["text"] if isinstance(resp, dict) else resp.output[0].content[0]["text"]
    trace_id = mlflow.get_last_active_trace_id()
    trace = mlflow.get_trace(trace_id)
    print(f"\n=== {txn}  latency={time.time()-t0:.1f}s  trace_id={trace_id} ===")
    print(text[:700] + (" ..." if len(text) > 700 else ""))
    print(f"\n{'tool span':<24} {'implementation':<42} {'status':<8} {'ms':>8}  objects read")
    print("-" * 140)
    for s in trace.data.spans:
        if not str(s.span_type).upper().endswith("TOOL"):
            continue
        out = s.outputs
        if hasattr(out, "content"): out = out.content
        if isinstance(out, dict) and "content" in out: out = out["content"]
        try:
            payload = json.loads(out) if isinstance(out, str) else {}
        except Exception:
            payload = {}
        st = str(s.status.status_code).split(".")[-1]
        row = (txn, s.name, payload.get("_implementation", "(no provenance)"), st, (s.end_time_ns - s.start_time_ns) / 1e6, ", ".join(payload.get("_objects_read", [])))
        results.append(row)
        print(f"{row[1]:<24} {row[2]:<42} {row[3]:<8} {row[4]:>8.1f}  {row[5]}")

cases_after = case_ids()
tools = [r for r in results]
print(f"\ntool spans: {len(tools)}; tools called: {sorted({r[1] for r in tools})}")
assert tools, "no tool spans found"
assert all(r[2] == "agent_python:postgres:fraud_triage_agent" for r in tools), "a tool ran outside Postgres governance"
assert all(r[3] in ("OK", "UNSET") for r in tools), "a tool span has an error status"
assert "create_fraud_case" not in {r[1] for r in tools}, "create_fraud_case was called"
assert cases_before == cases_after, f"case set changed: {cases_before} -> {cases_after}"
print(f"cases unchanged: {cases_after}")
print(f"PASS: @champion v{champ.version} - every tool span ran as agent_python:postgres:fraud_triage_agent with no errors; no case created.")

run_at_utc: 2026-11-01T19:36:48+00:00
@champion -> v3 (run d25dfe72e129478a8e17c0d6dbc7d433)


2026/11/01 19:36:52 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's... I\'ll go from there.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(



=== TXN-0234209  latency=13.9s  trace_id=tr-764034ca8a1e1ae9b19cbc35d413a50a ===
# TXN-0234209: why it was flagged

I only explained this transaction. I didn't open, update, or close any case.

**Transaction:** $323.96 on 2026-06-27 for CUST-08669. The risk score is **0.984**.

## Why it was flagged
The three top signals all push the score up:
- **New device (SHAP +2.57):** This is the strongest signal. The customer normally uses 5 devices in 30 days, and this one was new.
- **Distance from home (SHAP +1.87):** The transaction was 450.6 km from the customer's home region of US-East.
- **Online transaction (SHAP +1.20):** Online transactions score as higher risk.

## Recommended action
**Block.**

## Confidence
**Medium (observed precision 0.50 in block_band, n=288).** Hi ...

tool span                implementation                             status         ms  objects read
----------------------------------------------------------------------------------------------------------------

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d32adad-a55b-4953-ba99-1953b044617b/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...an override, tell me.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(



=== TXN-0112255  latency=12.2s  trace_id=tr-a54c80e15e88d7fb7a5e3464f173695f ===
# TXN-0112255: why it was flagged

I only explained this transaction. I did not open, update or close any case.

**Transaction:** $169.69 on 2026-09-23 for CUST-09364. Risk score is **0.992**.

## Why it was flagged
The three top signals are:
1. **New device (SHAP +1.94).** The transaction came from a device not seen before. The customer used 5 devices in the last 30 days.
2. **Distance from home (SHAP +1.85).** It happened 332.9 km from the customer's home region, US-East.
3. **High-risk segment (SHAP +0.86).** The customer is in the "high" risk segment.

## Recommended action
**Block.**

## Confidence
**Medium (observed precision 0.50 in block_band, n=288).** Historically only about half o ...

tool span                implementation                             status         ms  objects read
----------------------------------------------------------------------------------------------------------------

[Trace(trace_id='tr-764034ca8a1e1ae9b19cbc35d413a50a'), Trace(trace_id='tr-a54c80e15e88d7fb7a5e3464f173695f')]